Para cada notebook tengo que descargarme de nuevo los ficheros comprimidos

In [ ]:
import os

url = 'http://dihana.cps.unizar.es/~cadrete/asr/musan_small.zip'
if not os.path.exists('musan_small.zip'):
    os.system('wget ' + url)
    os.system('unzip musan_small.zip')

url = 'http://dihana.cps.unizar.es/~cadrete/asr/rirs_noises_small.zip'
if not os.path.exists('rirs_noises_small.zip'):
    os.system('wget ' + url)
    os.system('unzip rirs_noises_small.zip')

url = 'https://dihana.unizar.es/~cadrete/valencia2526/fechas2.zip'
if not os.path.exists('fechas2.zip'):
    os.system('wget ' + url)
    os.system('unzip fechas2.zip')

--2025-12-20 21:49:42--  http://dihana.cps.unizar.es/~cadrete/asr/musan_small.zip
Resolving dihana.cps.unizar.es (dihana.cps.unizar.es)... 155.210.153.34
Connecting to dihana.cps.unizar.es (dihana.cps.unizar.es)|155.210.153.34|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 481340505 (459M) [application/zip]
Saving to: ‘musan_small.zip’

     0K .......... .......... .......... .......... ..........  0%  220K 35m32s
    50K .......... .......... .......... .......... ..........  0%  417K 27m10s
   100K .......... .......... .......... .......... ..........  0%  184M 18m7s
   150K .......... .......... .......... .......... ..........  0%  457K 17m53s
   200K .......... .......... .......... .......... ..........  0% 81.5M 14m19s
   250K .......... .......... .......... .......... ..........  0%  139M 11m56s
   300K .......... .......... .......... .......... ..........  0%  200M 10m14s
   350K .......... .......... .......... .......... ..........  0%  187M 8m5

Archive:  musan_small.zip
   creating: musan_small/
   creating: musan_small/music_small/
  inflating: musan_small/music_small/music-jamendo-0029.wav  
  inflating: musan_small/music_small/music-rfm-0095.wav  
  inflating: musan_small/music_small/music-rfm-0111.wav  
  inflating: musan_small/music_small/music-jamendo-0127.wav  
  inflating: musan_small/music_small/music-rfm-0108.wav  
  inflating: musan_small/music_small/music-rfm-0029.wav  
  inflating: musan_small/music_small/music-hd-0050.wav  
  inflating: musan_small/music_small/music-fma-0049.wav  
  inflating: musan_small/music_small/music-fma-0068.wav  
  inflating: musan_small/music_small/music-fma-0087.wav  
  inflating: musan_small/music_small/music-hd-0043.wav  
  inflating: musan_small/music_small/music-rfm-0007.wav  
  inflating: musan_small/music_small/music-fma-wa-0025.wav  
  inflating: musan_small/music_small/music-jamendo-0188.wav  
  inflating: musan_small/music_small/music-jamendo-0071.wav  
  inflating: musan_smal

--2025-12-20 21:50:10--  http://dihana.cps.unizar.es/~cadrete/asr/rirs_noises_small.zip
Resolving dihana.cps.unizar.es (dihana.cps.unizar.es)... 155.210.153.34
Connecting to dihana.cps.unizar.es (dihana.cps.unizar.es)|155.210.153.34|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67032603 (64M) [application/zip]
Saving to: ‘rirs_noises_small.zip’

     0K .......... .......... .......... .......... ..........  0%  159K 6m51s
    50K .......... .......... .......... .......... ..........  0%  249K 5m37s
   100K .......... .......... .......... .......... ..........  0%  250K 5m12s
   150K .......... .......... .......... .......... ..........  0%  204M 3m54s
   200K .......... .......... .......... .......... ..........  0%  239K 4m1s
   250K .......... .......... .......... .......... ..........  0%  233K 4m8s
   300K .......... .......... .......... .......... ..........  0%  225K 4m13s
   350K .......... .......... .......... .......... ..........  0%  277M 3

In [ ]:
!pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.1 MB/s eta 0:00:0000:010:01


# Ejercicio 2.1

En este ejercicio hay que crear un nuevo tokenizador pero teniendo en cuenta que hay que añadir 4 nuevos tokens especiales (transcribe_es, transcribe_en, translate_es_en, translate_en_es)

En la funcion encode se le añade un nuevo parámetro "action" para añadir uno de los 4 tokens especiales delante del texto

En la función decode se reemplaza los tokens especiales por la cadena vacia

In [ ]:
import torch
import pandas as pd

def get_tokens_csv(files: list[str]):
    """
    Construye un vocabulario a partir de una lista de ficheros CSV.
    :param files: Lista de ficheros CSV para construir el vocabulario.
    :type files: list[str]
    :return: Diccionario que mapea tokens a índices.
    :rtype: dict
    """
    tokens = {}
    for file in files:
        df = pd.read_csv(file)
        for frase in df['txt']:
            for token in frase.split():
                if token not in tokens:
                    tokens[token] = len(tokens)
    return tokens

class Tokenizer():
    def __init__(self, files: list[str]):

        assert len(files) > 0, "Debe proporcionar al menos un fichero para construir el vocabulario."

        self.word2index = get_tokens_csv(files)
        # añadimos tokens especiales
        self.translate_tokens = ['<transcribe_es>', '<transcribe_en>', '<translate_en_es>', '<translate_es_en>']
        special_tokens = ['<pad>', '<sos>', '<eos>'] + self.translate_tokens
        for token in special_tokens:
            if token not in self.word2index:
                self.word2index[token] = len(self.word2index)
        self.index2word = {v:k for k,v in self.word2index.items()}

    def encode(self, x, seq_len=-1, action=None):
        """
        Codifica una cadena de texto en una secuencia de índices.
        El parámetro action permite añadir un token especial al inicio de la secuencia.
        """
        if action is not None and action in self.translate_tokens:
            x = action + ' ' + x
        x = '<sos> ' + x + ' <eos>' # añadimos tokens de inicio y fin
        x = [self.word2index[w] for w in x.split()]
        if seq_len > len(x):
            x = x + [self.word2index['<pad>']] * (seq_len - len(x)) # relleno con token padding
        return torch.tensor(x)

    def decode(self, x):
        """
        Decodifica una secuencia de índices en una cadena de texto.
        Para los nuevos tokens especiales de traducción, los elimina del resultado convirtiendolos en cadenas vacías.
        """
        if isinstance(x, torch.Tensor):
            x = x.tolist()
        x = ' '.join([self.index2word[i] for i in x])
        x = x.replace('<sos>', '').replace('<eos>', '').replace('<pad>', '')
        for token in self.translate_tokens:
            x = x.replace(token, '')
        return x.strip()

En la celda siguiente creamos 4 dataframes distintos que son los siguientes:
1. Contiene transcripciones del audio español a texto en español
2. Contiene traducciones del audio en español al texto en inglés
3. Contiene transcripciones del audio en inglés al texto en inglés
4. Contiene traducciones del audio en inglés al texto en español


In [ ]:
files = {
    'es_train': 'fechas2/fechas2_train.es.csv',
    'es_test': 'fechas2/fechas2_test.es.csv',
    'en_train': 'fechas2/fechas2_train.en.csv',
    'en_test': 'fechas2/fechas2_test.en.csv'
}

def load_data():
    data_frames = []

    df_es_train = pd.read_csv(files['es_train'])
    df_es_test = pd.read_csv(files['es_test'])
    df_en_train = pd.read_csv(files['en_train'])
    df_en_test = pd.read_csv(files['en_test'])

    # primero transcripción del archivo en español
    df_es_to_es = pd.concat([df_es_train, df_es_test], ignore_index=True)
    df_es_to_es['action'] = '<transcribe_es>'
    data_frames.append(df_es_to_es)

    # ahora del español al inglés
    df_en = pd.concat([df_en_train, df_en_test], ignore_index=True)
    df_es_to_en = pd.concat([df_es_train, df_es_test], ignore_index=True)
    df_es_to_en['txt'] = df_en['txt'] # cambiamos la columna txt por la del inglés
    df_es_to_en['action'] = '<translate_es_en>'
    data_frames.append(df_es_to_en)

    # transcripción del archivo en inglés
    df_en_to_en = pd.concat([df_en_train, df_en_test], ignore_index=True)
    df_en_to_en['action'] = '<transcribe_en>'
    data_frames.append(df_en_to_en)

    # ahora del inglés al español
    df_es = pd.concat([df_es_train, df_es_test], ignore_index=True)
    df_en_to_es = pd.concat([df_en_train, df_en_test], ignore_index=True)
    df_en_to_es['txt'] = df_es['txt'] # cambiamos la columna txt por la del español
    df_en_to_es['action'] = '<translate_en_es>'
    data_frames.append(df_en_to_es)

    return data_frames

dataframes = load_data()
# Mostrar las primeras filas de cada DataFrame
print(dataframes[0].head())
print(dataframes[1].head())
print(dataframes[2].head())
print(dataframes[3].head())

                                   wav                            txt  \
0  fechas2/train/fechas2_000000.es.wav  por favor el siguiente jueves   
1  fechas2/train/fechas2_000001.es.wav           el viernes que viene   
2  fechas2/train/fechas2_000002.es.wav             el viernes gracias   
3  fechas2/train/fechas2_000003.es.wav            el martes por favor   
4  fechas2/train/fechas2_000004.es.wav                         mañana   

            action  
0  <transcribe_es>  
1  <transcribe_es>  
2  <transcribe_es>  
3  <transcribe_es>  
4  <transcribe_es>  
                                   wav                   txt  \
0  fechas2/train/fechas2_000000.es.wav  please next thursday   
1  fechas2/train/fechas2_000001.es.wav           next friday   
2  fechas2/train/fechas2_000002.es.wav      friday thank you   
3  fechas2/train/fechas2_000003.es.wav        tuesday please   
4  fechas2/train/fechas2_000004.es.wav              tomorrow   

              action  
0  <translate_es_en>  
1  <

In [ ]:
import torchaudio
import glob
import numpy as np
import scipy

class NoiseAug(object):
    def __init__(self, noise_dir='musan_small/', prob=0.5):
        self.prob = prob
        self.noises = glob.glob(noise_dir+'/*/*.wav')

    def __call__(self, x):
        if np.random.uniform() < self.prob:
            n = torchaudio.load( np.random.choice(self.noises) )[0][0]
            if len(n) < len(x):
                n = torch.nn.functional.pad(n, (0, len(x)-len(n)), value=0)
            elif len(n) > len(x):
                t0 = np.random.randint(0, len(n) - len(x))
                n = n[t0:t0+len(x)]
            n = n.numpy()
            p_x = x.std()**2
            p_n = n.std()**2
            snr = np.random.uniform(5, 15)
            n = n * np.sqrt(p_x/p_n) * np.power(10, -snr/20)
            x = x + n
        return x

class RIRAug(object):
    def __init__(self, rir_dir='RIRS_NOISES_small/simulated_rirs_small/', prob=0.5):
        self.prob = prob
        self.rirs = glob.glob(rir_dir+'/*.wav')

    def __call__(self, x):
        if np.random.uniform() < self.prob:
            n = len(x)
            rir = torchaudio.load( np.random.choice(self.rirs) )[0][0]
            rir = rir.numpy()
            rir = rir / np.max(np.abs(rir))
            x = scipy.signal.convolve(x, rir)
            t0 = np.argmax(np.abs(rir))
            x = x[t0:t0+n]
        return x


Creamos el dataset nuevo que se le pasa la lista de dataframes definida anteriormente.
Cuando se realiza la función getitem codificamos la cadena con la acción definida

In [ ]:
def identity(x):
    return x

class DatasetEjercicio2(torch.utils.data.Dataset):
    def __init__(self, dataframes: list[pd.DataFrame], tokenizer: Tokenizer, audio_len=4*16000, transform=[identity]):
        assert len(dataframes) > 0, "Debe proporcionar al menos un DataFrame para construir el vocabulario."

        # concatenamos todos los dataframes en uno solo porque tienen la misma estructura
        self.data = pd.concat(dataframes, ignore_index=True)

        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, fs = torchaudio.load(self.data.iloc[idx]['wav'])
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]

        x = x[0].numpy()
        for t in self.transform:
            x = t(x)

        text = self.data.iloc[idx]['txt']
        action = self.data.iloc[idx]['action']
        y = self.tokenizer.encode(text, action=action)
        return x, y

tokenizer = Tokenizer([files['es_train'], files['es_test'], files['en_train'], files['en_test']])
dataframes = load_data()
dataset = DatasetEjercicio2(dataframes, tokenizer)

In [ ]:
print(tokenizer.word2index)

{'por': 0, 'favor': 1, 'el': 2, 'siguiente': 3, 'jueves': 4, 'viernes': 5, 'que': 6, 'viene': 7, 'gracias': 8, 'martes': 9, 'mañana': 10, 'este': 11, 'pasado': 12, 'lunes': 13, 'próximo': 14, 'en': 15, 'tres': 16, 'días': 17, 'un': 18, 'par': 19, 'de': 20, 'miércoles': 21, 'please': 22, 'next': 23, 'thursday': 24, 'friday': 25, 'thank': 26, 'you': 27, 'tuesday': 28, 'tomorrow': 29, 'this': 30, 'coming': 31, 'thanks': 32, 'the': 33, 'day': 34, 'after': 35, 'monday': 36, 'in': 37, 'three': 38, 'days': 39, 'you.': 40, 'a': 41, 'couple': 42, 'of': 43, 'wednesday': 44, 'following': 45, 'on': 46, '<pad>': 47, '<sos>': 48, '<eos>': 49, '<transcribe_es>': 50, '<transcribe_en>': 51, '<translate_en_es>': 52, '<translate_es_en>': 53}


Ejemplos de los audios con el texto

In [ ]:
trainset = DatasetEjercicio2(dataframes=dataframes, tokenizer=tokenizer, transform=[NoiseAug()])
x, y = trainset[40000]

print(tokenizer.decode(y))
from IPython.display import Audio
Audio(x,rate=16000)

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

en un par de días gracias


In [ ]:
trainset = DatasetEjercicio2(dataframes=dataframes, tokenizer=tokenizer, transform=[NoiseAug()])
x, y = trainset[20000]

print(tokenizer.decode(y))
from IPython.display import Audio
Audio(x,rate=16000)

the following monday thanks


In [ ]:
trainset = DatasetEjercicio2(dataframes=dataframes, tokenizer=tokenizer, transform=[NoiseAug()])
x, y = trainset[10000]

print(tokenizer.decode(y))
from IPython.display import Audio
Audio(x,rate=16000)

pasado mañana gracias


In [ ]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model=512, d_ff=1024, dropout=0.1, **kwargs):
        super().__init__()
        self.ff = torch.nn.Sequential(
            torch.nn.LayerNorm(d_model),
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.ff(x)

class SelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

    def forward(self, x):
        x = self.norm(x)
        b = x.shape[0]
        q = self.q_linear(x).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).view(b, -1, self.n_heads, self.d_head)

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale

        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out

class Encoder(torch.nn.Module):
    def __init__(self, nb_layers=6, **kwargs):
        super().__init__()
        self.seq_len = kwargs['seq_len']
        self.pos = torch.nn.Parameter(torch.randn(1, self.seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([SelfAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])

    def forward(self, x):
        b, t, d = x.shape
        x = x + self.pos[:, :t, :]
        for att, ff in zip(self.att, self.ff):
            x = x + att(x)
            x = x + ff(x)
        return x


class CausalSelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.seq_len = kwargs['seq_len']
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

        self.register_buffer("mask", torch.tril(torch.ones(self.seq_len, self.seq_len))[None, None, ...] == 0)


    def forward(self, x):
        x = self.norm(x)
        b, n, d = x.shape
        q = self.q_linear(x).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).view(b, -1, self.n_heads, self.d_head)

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale

        scores = scores.masked_fill(self.mask[:,:,:n,:n], float('-inf'))
        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out

class CrossAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

    def forward(self, x1, x2):
        x1 = self.norm1(x1)
        x2 = self.norm2(x2)
        b = x1.shape[0]
        q = self.q_linear(x1).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x2).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x2).view(b, -1, self.n_heads, self.d_head)

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale

        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out, att

class Decoder(torch.nn.Module):
    def __init__(self, nb_layers=6, **kwargs):
        super().__init__()
        self.seq_len = kwargs['seq_len']
        self.pos = torch.nn.Parameter(torch.randn(1, self.seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([CausalSelfAttention(**kwargs) for _ in range(nb_layers)])
        self.cross_att = torch.nn.ModuleList([CrossAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])

    def forward(self, x, enc):
        b, t, d = x.shape
        x = x + self.pos[:, :t, :]
        for att, cross_att, ff in zip(self.att, self.cross_att, self.ff):
            x = x + att(x)
            x = x + cross_att(x, enc)[0]
            x = x + ff(x)
        return x

class SpecAug(torch.nn.Module):
    def __init__(self, prob_t_warp=0.5,
                       t_factor=(0.9, 1.1),
                       f_mask_width = (0, 8),
                       t_mask_width = (0, 10),
                       nb_f_masks=[1,2],
                       nb_t_masks=[1,2] ):
        super().__init__()
        self.t_factor = t_factor
        self.f_mask_width = f_mask_width
        self.t_mask_width = t_mask_width
        self.nb_f_masks = nb_f_masks
        self.nb_t_masks = nb_t_masks
        self.prob_t_warp = prob_t_warp

    def time_warp(self, x):
        x = torch.nn.functional.interpolate(x, size=(int(x.shape[2]*np.random.uniform(*self.t_factor)), ))
        # print('warp', x.shape[2])
        return x

    def freq_mask(self, x):
        for _ in range(np.random.randint(*self.nb_f_masks)):
            f = np.random.randint(*self.f_mask_width)
            f0 = np.random.randint(0, x.shape[1]-f)
            # print('f', f0, f0+f)
            x[:,f0:f0+f,:] = 0
        return x

    def time_mask(self, x):
        for _ in range(np.random.randint(*self.nb_t_masks)):
            t = np.random.randint(*self.t_mask_width)
            t0 = np.random.randint(0, x.shape[2]-t)
            # print('t', t0, t0+t)
            x[:,:,t0:t0+t] = 0
        return x

    def forward(self, x):
        if np.random.uniform() < self.prob_t_warp:
            x = self.time_warp(x)
        x = self.freq_mask(x)
        x = self.time_mask(x)
        return x

En AudioTransformer, en la función generate tenemos que pasarle uno de los 4 tokens que se han creado para que el modelo sepa que tarea va a realizar.
El token se añade después del token sos

In [ ]:
class AudioFeatures(torch.nn.Module):
    def __init__(self, feat_dim=80, d_model=512, **kwargs):
        super().__init__()
        self.fe = torchaudio.transforms.MelSpectrogram(
                        n_fft=512,
                        win_length=25*16,
                        hop_length=10*16,
                        n_mels=feat_dim)                            # 25ms window, 10ms shift
        self.spec_aug = SpecAug()
        self.linear = torch.nn.Linear(feat_dim, d_model)

    def forward(self, x):
        x = self.fe(x)
        x = (x+1e-6).log()
        if self.training:
            x = self.spec_aug(x)
        x = x.transpose(1, 2)
        x = self.linear(x)
        return x

class AudioTransformer(torch.nn.Module):
    def __init__(self, vocab_size=24, **kwargs):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = kwargs['seq_len']

        self.fe = AudioFeatures(**kwargs)
        self.enc = Encoder(**kwargs)

        self.emb = torch.nn.Embedding(vocab_size, kwargs['d_model'])
        self.dec = Decoder(**kwargs)
        self.out = torch.nn.Linear(kwargs['d_model'], vocab_size)


    def encoder(self, x):
        x = self.fe(x)
        return self.enc(x)

    def decoder(self, y, enc):
        y = self.emb(y)
        dec = self.dec(y, enc)
        return self.out(dec)

    def forward(self, x, y):
        enc = self.encoder(x)
        return self.decoder(y, enc)

    def loss(self, x, y):
        logits = self(x, y[:,:-1])
        target = y[:,1:]
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, self.vocab_size),
                                                 target.reshape(-1), ignore_index=tokenizer.word2index['<pad>'])
        return loss


    def generate(self, x, action=None):
        device = next(self.parameters()).device
        self.eval()
        if action is not None:
            y = [tokenizer.word2index['<sos>'], action]
        else:
            y = [tokenizer.word2index['<sos>'],]
        with torch.no_grad():
            enc = self.encoder(x.to(device))

            while y[-1] != tokenizer.word2index['<eos>'] and len(y) < self.seq_len:
                logits = self.decoder(torch.tensor(y).unsqueeze(0).to(device), enc)
                y.append(logits.argmax(-1)[:,-1].item())

        return y

trainset = DatasetEjercicio2(dataframes=dataframes, tokenizer=tokenizer, transform=[NoiseAug(), RIRAug()])


# Collate function to pad sequences to the same length in a batch
def collate_fn(batch):
    x_batch = [torch.tensor(x, dtype=torch.float32) for x, y in batch]
    y_batch = [y for x, y in batch]

    # Pad audio to same length
    x_batch = torch.nn.utils.rnn.pad_sequence([x for x in x_batch], batch_first=True, padding_value=0.0)

    # Pad text sequences to same length
    y_batch = torch.nn.utils.rnn.pad_sequence([y for y in y_batch], batch_first=True, padding_value=tokenizer.word2index['<pad>'])

    return x_batch, y_batch

Como la cantidad de datos de entrenamiento utilizados para este problema es más grande que en los otros ejercicios, he reducido el número de épocas a 2

In [ ]:
model = AudioTransformer(vocab_size=len(tokenizer.word2index), d_model=256, nb_layers=8, n_heads=8, d_head=32, dropout=0.1, seq_len=500)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

nb_epochs = 2
batch_size = 32
model.train()

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
for e in range(nb_epochs):
    avg_loss = 0
    for x, y in trainloader:
        x = x.to(device)
        y = y.to(device)
        opt.zero_grad()
        loss = model.loss(x, y)
        loss.backward()
        opt.step()
        avg_loss += loss.item()
    print('epoch %d/%d: avg_loss: %.2f' % (e,nb_epochs,avg_loss/len(trainloader)))

torch.save([model, opt], 'model21.pt')

epoch 0/2: avg_loss: 0.76
epoch 1/2: avg_loss: 0.32


In [ ]:
class TestSet(torch.utils.data.Dataset):
    def __init__(self, csv: str, tokenizer: Tokenizer, audio_len=4*16000, transform=[identity]):
        self.data = pd.read_csv(csv)

        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, fs = torchaudio.load(self.data.iloc[idx]['wav'])
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]

        x = x[0].numpy()
        for t in self.transform:
            x = t(x)

        # obtener transcripcion
        text = self.data.iloc[idx]['txt']
        action = self.data.iloc[idx]['action']
        action = "<" + action + ">" # en el dataset de test los actions no tienen los <>
        y = self.tokenizer.encode(text, action=action)
        # print(x.shape, x.dtype)
        return x, y

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:0000:0100:01


In [ ]:
#[model, opt] = torch.load('model21.pt')
eos = tokenizer.word2index['<eos>']
testset = TestSet(csv="fechas2/fechas2_test_instruct.csv", tokenizer=tokenizer)
import jiwer
model.eval()
hyp_ = []
ref_ = []
for i,(x, y) in enumerate(testset):
    x = torch.tensor(x, dtype=torch.float32)
    x = x.to(device)
    action_token = y[1].item()
    y_pred = model.generate(x[None,...], action=action_token)

    hyp = ' '.join([str(i) for i in y_pred[1:-1]])
    y = y.numpy().tolist()
    # find the first 24 <eos> in list y
    y = y[:y.index(eos)]
    ref = ' '.join([str(i) for i in y[1:]])

    hyp_.append(hyp)
    ref_.append(ref)

out = jiwer.process_words(ref_, hyp_)
print(f"WER: {out.wer:.2%}")

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

WER: 6.23%


In [ ]:
x, y = testset[611]
print(tokenizer.decode(y))

output = model.generate(torch.tensor(x, dtype=torch.float32).unsqueeze(0), action=y[1].item())
print(tokenizer.decode(output))

Audio(x,rate=16000)

in a couple of days thanks
in a couple of days thanks


In [ ]:
x, y = testset[876]
print(tokenizer.decode(y))

output = model.generate(torch.tensor(x, dtype=torch.float32).unsqueeze(0), action=y[1].item())
print(tokenizer.decode(output))

Audio(x,rate=16000)

the day after tomorrow thanks
the day after tomorrow thanks


In [ ]:
x, y = testset[0]
print(tokenizer.decode(y))

output = model.generate(torch.tensor(x, dtype=torch.float32).unsqueeze(0), action=y[1].item())
print(tokenizer.decode(output))

Audio(x,rate=16000)